In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [ ]:
INPUT_PATH = "stage1_outputs/prepared_stage1_dataset.csv"
OUTPUT_DIR = Path("stage3_model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
CLASSIFICATION_THRESHOLD = 0.5

In [ ]:
def load_data(path):
    df = pd.read_csv(path)

    for col in ["buyout_flag", "outcome_unknown", "lifecycle_incomplete"]:
        if col in df.columns:
            vals = (
                df[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .replace({"<na>": None, "nan": None, "": None})
            )
            df[col] = vals.map({
                "true": True,
                "false": False,
                "1": True,
                "0": False
            }).astype("boolean")

    numeric_cols = [
        "speed_to_delivery_days",
        "days_sale_to_handed",
        "days_handed_to_issued_pvz",
        "days_to_outcome",
        "lead_price",
        "order_amount",
        "delivery_cost"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def prepare_model_dataset(df):
    stats = {"rows_input": int(len(df))}

    if "lifecycle_incomplete" in df.columns:
        bad = df["lifecycle_incomplete"].astype("boolean").fillna(False)
        stats["excluded_lifecycle_incomplete"] = int(bad.sum())
        df = df.loc[~bad].copy()
    else:
        stats["excluded_lifecycle_incomplete"] = 0

    if "outcome_unknown" in df.columns:
        unknown = df["outcome_unknown"].astype("boolean").fillna(False)
        stats["excluded_outcome_unknown"] = int(unknown.sum())
        df = df.loc[~unknown].copy()
    else:
        stats["excluded_outcome_unknown"] = 0

    df = df.loc[df["buyout_flag"].notna()].copy()
    df["is_buyout"] = df["buyout_flag"].astype(int)

    feature_candidates = [
        "region",
        "delivery_service",
        "delivery_tariff",
        "payment_type",
        "delivery_method",
        "manager_id",
        "delivery_manager_id",
        "manager_group",
        "lead_qualification",
        "lead_pipeline_id",
        "speed_to_delivery_days"
    ]

    feature_cols = [c for c in feature_candidates if c in df.columns]

    model_df = df[["lead_id", "is_buyout"] + feature_cols].copy()

    cat_cols = [c for c in feature_cols if model_df[c].dtype == object]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for c in cat_cols:
        model_df[c] = model_df[c].fillna("Не указан")

    stats["rows_final"] = int(len(model_df))
    stats["buyout_rate"] = float(model_df["is_buyout"].mean()) if len(model_df) else None
    stats["feature_count"] = len(feature_cols)

    return model_df, feature_cols, cat_cols, num_cols, stats


df_raw = load_data(INPUT_PATH)
model_df, feature_cols, cat_cols, num_cols, prep_stats = prepare_model_dataset(df_raw)

print(json.dumps(prep_stats, ensure_ascii=False, indent=2))
print("Feature columns:", feature_cols)
print("Shape:", model_df.shape)

model_df.head(3)

In [ ]:
X = model_df[feature_cols]
y = model_df["is_buyout"]

X_train, X_test, y_train, y_test, lead_train, lead_test = train_test_split(
    X,
    y,
    model_df["lead_id"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

split_info = {
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_buyout_rate": float(y_train.mean()),
    "test_buyout_rate": float(y_test.mean()),
}
print(json.dumps(split_info, ensure_ascii=False, indent=2))

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols)
])

In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "GradientBoosting": Pipeline([
        ("prep", preprocessor),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
}

In [ ]:
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

list(fitted_models.keys())

In [ ]:
def evaluate_model(model, X_test, y_test, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "pred_proba": proba,
        "pred_class": pred
    }

eval_results = {}
rows = []

for name, model in fitted_models.items():
    res = evaluate_model(model, X_test, y_test, threshold=CLASSIFICATION_THRESHOLD)
    eval_results[name] = res
    rows.append({
        "model": name,
        "roc_auc": res["roc_auc"],
        "pr_auc": res["pr_auc"],
        "accuracy": res["accuracy"],
        "precision": res["precision"],
        "recall": res["recall"],
        "f1": res["f1"]
    })

metrics_df = pd.DataFrame(rows).sort_values(["roc_auc", "pr_auc"], ascending=False)
metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
metrics_df

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_proba = eval_results[best_model_name]["pred_proba"]
best_pred = eval_results[best_model_name]["pred_class"]

print("Лучшая модель:", best_model_name)

In [ ]:
plt.figure(figsize=(10, 7))

for name, res in eval_results.items():
    fpr, tpr, _ = roc_curve(y_test, res["pred_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

for name, res in eval_results.items():
    precision, recall, _ = precision_recall_curve(y_test, res["pred_proba"])
    plt.plot(recall, precision, label=f"{name} (PR-AUC={res['pr_auc']:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "precision_recall_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.hist(best_proba[y_test.values == 1], bins=30, alpha=0.7, label="Выкуп = 1")
plt.hist(best_proba[y_test.values == 0], bins=30, alpha=0.7, label="Выкуп = 0")
plt.xlabel("Predicted probability")
plt.ylabel("Количество заказов")
plt.title(f"Распределение вероятностей: {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "probability_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
cm = confusion_matrix(y_test, best_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual_0", "Actual_1"],
    columns=["Pred_0", "Pred_1"]
)
cm_df.to_csv(OUTPUT_DIR / "best_model_confusion_matrix.csv")
cm_df

In [ ]:
pred_df = pd.DataFrame({
    "lead_id": lead_test.values,
    "y_true": y_test.values,
    "buyout_probability": best_proba,
    "predicted_class": best_pred,
    "model_name": best_model_name
})

bins = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
labels = ["0–0.5", "0.5–0.7", "0.7–0.8", "0.8–0.9", "0.9–0.95", "0.95–1.0"]

pred_df["score_bin"] = pd.cut(
    pred_df["buyout_probability"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

score_bin_table = (
    pred_df.groupby("score_bin", observed=False)
    .agg(
        orders=("lead_id", "count"),
        actual_buyout_rate=("y_true", "mean")
    )
    .reset_index()
)

score_bin_table["actual_buyout_rate_pct"] = score_bin_table["actual_buyout_rate"] * 100

score_bin_table.to_csv(OUTPUT_DIR / "score_bin_performance.csv", index=False)
score_bin_table

In [ ]:
pred_df.to_csv(OUTPUT_DIR / "buyout_predictions.csv", index=False)
pred_df.head()

In [ ]:
business_summary = {
    "rows_modelled": int(len(model_df)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_buyout_rate": float(y_train.mean()),
    "test_buyout_rate": float(y_test.mean()),
    "best_model": best_model_name,
    "best_roc_auc": float(metrics_df.iloc[0]["roc_auc"]),
    "best_pr_auc": float(metrics_df.iloc[0]["pr_auc"]),
    "best_accuracy": float(metrics_df.iloc[0]["accuracy"]),
    "best_precision": float(metrics_df.iloc[0]["precision"]),
    "best_recall": float(metrics_df.iloc[0]["recall"]),
    "best_f1": float(metrics_df.iloc[0]["f1"]),
}

with open(OUTPUT_DIR / "model_business_summary.json", "w", encoding="utf-8") as f:
    json.dump(business_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(business_summary, ensure_ascii=False, indent=2))